# Phase 3 : Benchmark machine learning contre le tarif GLM

**Objectif.** Mesurer ce qu'un modèle de gradient boosting (LightGBM) apporte au tarif GLM v0.2.0 : sur les mêmes polices de test, avec les mêmes métriques, à niveau de prime égal. La question n'est pas "le GBM fait-il mieux ?" mais "où, de combien, et à quel prix en lisibilité ?".

**Le tarif à battre.** Pour une police $i$, on note :
- $e_i$ son exposition en années, plafonnée à 1 ;
- $\hat\lambda_i$ la fréquence annuelle de sinistres prédite par le GLM Poisson, c'est-à-dire le nombre attendu de sinistres pour une exposition d'un an ;
- $\hat\mu_i^{ecr}$ le coût moyen d'un sinistre indemnisé, écrêté au 99e percentile (16 282 EUR), prédit par le GLM Gamma ;
- $\tau_{obs} = 0.7317$ la proportion de sinistres déclarés qui donnent lieu à un montant ;
- $\tau_{graves} = 0.4529$ le taux de rechargement des sinistres graves, tel que $1 + \tau_{graves}$ ramène la charge écrêtée à la charge totale observée sur le train ;
- $k = 0.994665$ la constante de recalage du niveau global de prime sur le train.

La prime pure annuelle prédite vaut

$$\widehat{PP}_i = k \times \hat\lambda_i \times \tau_{obs} \times \hat\mu_i^{ecr} \times (1 + \tau_{graves})$$

et la charge attendue de la police $i$ vaut $e_i \times \widehat{PP}_i$.

| Indicateur (test) | Valeur GLM |
|---|---|
| Indice de Gini (prime pure) | 0.3592 |
| Lift G10/G1 fréquence | 4.52 |
| D2 fréquence (déviance Poisson) | 0.0357 |
| D2 sévérité (déviance Gamma) | 0.0034 |
| D2 prime pure (déviance Tweedie, p = 1.667) | 0.0519 |
| Calibration A/E train / test | 1.0000 / 0.9531 |
| Amplitude des primes (99e / 1er pct) | 11.9 |

**Règles du benchmark.**
1. *Même découpage, test sanctuarisé.* Aucun choix (hyperparamètres, nombre d'arbres, variables) ne s'appuie sur le test. Tout réglage passe par une validation croisée sur le train.
2. *Mêmes métriques, même code.* Les indicateurs du GLM sont recalculés ici par les fonctions qui évalueront le GBM.
3. *Même niveau de prime.* Le GBM n'a pas la propriété d'équilibre du GLM Poisson à lien canonique. Il est donc recalé sur le train avant toute comparaison.

**Questions à trancher.** L'interaction âge × bonus-malus est-elle un gain réel ou un artefact ? Le résidu non linéaire sur `log_Density` compte-t-il en pratique ? Le plafond de la sévérité vient-il des forfaits ou de la classe de modèles ?

## Journal des décisions (suite de la Phase 2)

| N° | Constat | Décision | Justification | Impact |
|---|---|---|---|---|
| 39 | Le benchmark n'a de sens que si les deux modèles sont jugés sur les mêmes polices | Réutiliser `split_train_test.parquet` tel quel et ne consulter le test qu'à l'évaluation finale | Un réglage guidé par le test (early stopping, hyperparamètres) surestime le GBM, alors que le GLM n'a jamais vu le test | Tout réglage du GBM passe par validation croisée sur le train |
| 40 | LightGBM et XGBoost offrent tous deux les objectifs Poisson, Gamma, Tweedie et un offset | Retenir LightGBM seul | Arbres par histogrammes rapides sur 542 397 lignes, catégorielles natives, empreinte mémoire faible (crash de kernel en Phase 2) | Le récit reste GLM contre GBM, pas un concours de librairies |
| 41 | Le tarif v0.2.0 est figé dans `models/tarif_glm_phase2.json` | Scorer depuis ce fichier plutôt que réajuster les GLM | On compare au tarif tagué, pas à une réestimation, et la même fonction de scoring servira à l'appli en Phase 5 | Fonction de scoring à écrire dans `src/` |
| 42 | `BonusMalusBand` et `VehPowerGrp` sont absentes du parquet et leurs bornes absentes du JSON du tarif | Centraliser toutes les définitions de regroupement dans `src/features.py` | Un tarif doit se calculer à partir des seules variables brutes, et un même code de regroupement évite que notebook et appli appliquent des bornes différentes | Le notebook 03 et l'appli Streamlit importeront la même fonction |

## 1. Chargement des données et contrôles d'intégrité

Trois vérifications avant toute modélisation :
- **le découpage relu est exactement celui de la Phase 2** (542 397 / 135 616). Si un seul effectif diffère, les chiffres de référence du GLM ne sont plus comparables ;
- **les variables brutes sont présentes** (âge du conducteur, bonus-malus, densité, etc.) à côté des variables regroupées. Un GBM trouve lui-même ses seuils, et lui donner les tranches du GLM pourrait le brider. La question sera tranchée en section 3 ;
- **la structure du fichier de tarif GLM**, pour écrire la fonction de scoring.

On affiche aussi les totaux par échantillon (exposition, sinistres, charge) : ce sont les quantités que tout tarif recalé doit reproduire.

In [1]:
# =============================================================================
# Section 1 : chargement, controles d'integrite, structure du tarif GLM
# =============================================================================
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Racine du projet : on remonte jusqu'au dossier qui contient .git,
# le notebook fonctionne ainsi quel que soit le dossier de lancement
ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
SEED = 2026

# --- 1. Donnees nettoyees et decoupage fige (relu, jamais regenere) ---
df = pd.read_parquet(DATA_DIR / "fremtpl2_clean.parquet")
split = pd.read_parquet(DATA_DIR / "split_train_test.parquet")

# validate="one_to_one" leve une erreur si une police apparait deux fois
df = df.merge(split[["IDpol", "split"]], on="IDpol", how="left", validate="one_to_one")
del split
gc.collect()

assert df["split"].notna().all(), "Des polices n'ont pas de split"
effectifs = df["split"].value_counts()
assert sorted(effectifs.tolist()) == [135_616, 542_397], f"Decoupage inattendu :\n{effectifs}"
print(f"Polices : {len(df):,}")
print(effectifs.to_string(), "\n")

# --- 2. Totaux par echantillon : ce que tout tarif recale doit reproduire ---
resume = df.groupby("split").agg(
    polices=("IDpol", "size"),
    exposition=("Exposure", "sum"),
    sinistres=("ClaimNb", "sum"),
    sinistres_sev=("ClaimNbSev", "sum"),
    charge=("ClaimAmount", "sum"),
)
resume["frequence"] = resume["sinistres"] / resume["exposition"]
resume["prime_pure_obs"] = resume["charge"] / resume["exposition"]
print(resume.round(4).to_string(), "\n")

# --- 3. Inventaire des colonnes ---
print(df.dtypes.to_string(), "\n")

# Variables brutes : candidates pour le GBM (decision en section 3)
COLS_BRUTES = ["DrivAge", "VehAge", "VehPower", "BonusMalus",
               "Density", "Region", "VehBrand", "VehGas"]
manquantes = [c for c in COLS_BRUTES if c not in df.columns]
print("Variables brutes manquantes :", manquantes if manquantes else "aucune", "\n")

# --- 4. Structure du tarif GLM v0.2.0 (cles et types, pas toutes les valeurs) ---
with open(MODELS_DIR / "tarif_glm_phase2.json", encoding="utf-8") as f:
    tarif_glm = json.load(f)


def apercu_structure(obj, niveau=0, max_elements=5):
    """Affiche l'arborescence d'un JSON : cles, types, tailles, quelques valeurs."""
    marge = "    " * niveau
    if isinstance(obj, dict):
        # Toutes les cles au premier niveau, un apercu ensuite
        limite = len(obj) if niveau == 0 else max_elements
        for i, (cle, val) in enumerate(obj.items()):
            if i == limite:
                print(f"{marge}... ({len(obj) - limite} cles de plus)")
                break
            if isinstance(val, (dict, list)):
                print(f"{marge}{cle} : {type(val).__name__} de {len(val)} elements")
                apercu_structure(val, niveau + 1, max_elements)
            else:
                print(f"{marge}{cle} : {val!r}")
    elif isinstance(obj, list):
        for val in obj[:max_elements]:
            if isinstance(val, (dict, list)):
                apercu_structure(val, niveau + 1, max_elements)
            else:
                print(f"{marge}{val!r}")
        if len(obj) > max_elements:
            print(f"{marge}... ({len(obj) - max_elements} elements de plus)")


apercu_structure(tarif_glm)

Polices : 678,013
split
train    542397
test     135616 

       polices   exposition  sinistres  sinistres_sev       charge  frequence  prime_pure_obs
split                                                                                        
test    135616   71579.5329       7096           5253  11525316.88     0.0991        161.0141
train   542397  286780.5725      28960          21191  48383899.62     0.1010        168.7140 

IDpol               int64
ClaimNb             int64
Exposure          float64
Area             category
VehPower            int64
VehAge              int64
DrivAge             int64
BonusMalus          int64
VehBrand         category
VehGas                str
Density             int64
Region           category
ClaimAmount       float64
ClaimNbSev          int64
DrivAgeBand      category
VehAgeBand       category
log_Density       float64
RegionGrouped         str
split                 str 

Variables brutes manquantes : aucune 

version : 'phase2-glm-v1'
dat

### Lecture des résultats

**Intégrité.** 678 013 polices, découpage identique à celui de la Phase 2 (542 397 / 135 616), aucune police sans échantillon. Les chiffres de référence du GLM restent comparables.

**Le test est un échantillon plus "léger", pas le signe d'une mauvaise calibration.** Pour un échantillon $s \in \{train, test\}$, on note $E_s$ l'exposition totale en années, $N_s$ le nombre de sinistres, $N_s^{sev}$ le nombre de sinistres dotés d'un montant et $C_s$ la charge totale. La prime pure observée se décompose exactement en fréquence × taux d'observation × coût moyen :

$$\frac{C_s}{E_s} = \frac{N_s}{E_s} \times \frac{N_s^{sev}}{N_s} \times \frac{C_s}{N_s^{sev}}$$

| Composante | Train | Test | Ratio test / train |
|---|---|---|---|
| Fréquence $N_s / E_s$ | 0.1010 | 0.0991 | 0.982 |
| Taux d'observation $N_s^{sev} / N_s$ | 0.7317 | 0.7403 | 1.012 |
| Coût moyen $C_s / N_s^{sev}$ (EUR) | 2 283 | 2 194 | 0.961 |
| Prime pure $C_s / E_s$ (EUR) | 168.71 | 161.01 | 0.954 |

L'écart de prime pure (-4.6 %) vient surtout du coût moyen (-3.9 %). C'est attendu : la Phase 2 a établi que la charge a une variance théoriquement infinie ($\xi \approx 0.8$), donc la moyenne de 5 253 montants n'est pas stabilisée et quelques gros sinistres suffisent à la déplacer de plusieurs pourcents. Côté fréquence, si $N_{test}$ suit une loi de Poisson, son écart-type relatif vaut $1/\sqrt{\mathbb{E}[N_{test}]} \approx 1/\sqrt{7096} \approx 1.2\,\%$ : l'écart de -1.8 % représente environ 1.5 écart-type, ce qui reste compatible avec le hasard.

**Lien avec l'A/E test de 0.9531.** On note $e_i$ l'exposition de la police $i$ et $\bar\pi_s = \frac{1}{E_s}\sum_{i \in s} e_i \widehat{PP}_i$ la prime pure moyenne prédite sur l'échantillon $s$, de sorte que $A/E_s = C_s / (E_s\,\bar\pi_s)$. Comme $A/E_{train} = 1$ par construction de $k$, on obtient :

$$\frac{\bar\pi_{test}}{\bar\pi_{train}} = \frac{C_{test}/E_{test}}{C_{train}/E_{train}} \times \frac{1}{A/E_{test}} = \frac{0.9544}{0.9531} \approx 1.0013$$

Le GLM juge le portefeuille de test aussi risqué que celui du train (+0.1 %). L'écart vient des réalisations, pas du modèle. Ce calcul suppose que l'A/E de la Phase 2 rapporte la charge brute totale à la charge prédite : point à vérifier en section 2.

**Conséquence pour le benchmark.** Tout modèle recalé sur le train affichera un A/E test proche de 0.95. L'A/E global ne départagera donc pas le GLM et le GBM : on comparera leur calibration par déciles de prime prédite.

**Variables.** Toutes les variables brutes sont présentes, donc le GBM pourra les recevoir sous forme continue (décision en section 3). Deux points techniques :
- `VehGas` et `RegionGrouped` sont stockées en `str`, le type texte par défaut de pandas 3. LightGBM le refuse : il faudra les convertir explicitement en `category`.
- `BonusMalusBand` et `VehPowerGrp` n'existent pas dans le parquet : elles ont été construites dans le notebook 02.

**Le tarif GLM.** Le fichier contient bien les 53 + 28 coefficients et les quatre constantes de la Phase 2. Contrôle de non-fuite : $\tau_{obs} = 0.7317334$ vaut exactement 21 191 / 28 960, le ratio du train, donc la constante a été calculée sans voir le test. En revanche, le fichier n'est pas autosuffisant : il référence `BonusMalusBand` et `VehPowerGrp` sans en donner les bornes. Or un tarif doit pouvoir se calculer à partir des seules variables brutes d'un assuré, et c'est exactement ce que fera l'appli en Phase 5. Enfin, `performance_test` fournit les cibles de la section 2 avec toutes leurs décimales : la reproduction devra être exacte, pas seulement approchée.